# **Exercise 1: Duplicate Detection and Removal**

**Instructions**

**Objective:** Identify and remove duplicate entries in the Titanic dataset.  

+ Load the Titanic dataset.
  (https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/titanic%20dataset.zip)
+ Identify if there are any duplicate rows based on all columns.  
+ Remove any duplicate rows found in the dataset.  
+ Verify the removal of duplicates by checking the number of rows before and after the duplicate removal.  

**Hint:** Use the duplicated() and drop_duplicates() functions in Pandas.

In [ ]:
import pandas as pd
df = pd.read_csv('train.csv')
df.head()

In [ ]:
duplicates = df.duplicated().any()
print('Presence of duplicates:', duplicates)

## **Exercise 2: Handling Missing Values**

**Instructions**

+ Identify columns in the Titanic dataset with missing values.
+ Explore different strategies for handling missing data, such as removal, imputation, and filling with a constant value.
+ Apply each strategy to different columns based on the nature of the data.

**Hint:** Review methods like dropna(), fillna(), and SimpleImputer from scikit-learn.

In [ ]:
df = pd.read_csv('train.csv')
print('Empty cells per column:')
print(df.isnull().sum())

In [ ]:
total_rows = df.shape[0]
print('Total rows =', total_rows)

Column: Age  
Type: Numerical  
Method: Column drop 

In [ ]:
df.drop(columns=['Age'], inplace=True)

Column: Cabin  
Type: Categorical  
Method: Column drop

In [ ]:
df.drop(columns=['Cabin'], inplace=True)

Column: Embarked  
Type: Categorical  
Method: Mode Imputation

In [ ]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

As seen below: After applying the above strategies, there are no missing values left in the dataset.

In [ ]:
print(df.isnull().sum())

## **Exercise 3: Feature Engineering**
**Instructions**

+ Create new features, such as Family Size from SibSp and Parch, and Title extracted from the Name column.
+ Convert categorical variables into numerical form using techniques like one-hot encoding or label encoding.
+ You will encode new categorical features (like Title) here, but do not scale numerical features yet — that will come after outlier handling.

**Hint:** Utilize Pandas for data manipulation and scikit-learn’s preprocessing module for encoding.

**Using the following formula, we can create feature: 'Family Size'**  
Family Size = SibSp + Parch + 1, where one is the person themselves

In [ ]:
df['Family Size'] = df['SibSp'] + df['Parch'] + 1

Since title are usually followed by a dot in the name, we can extract the title from the 'Name' column using string manipulation techniques

In [ ]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

In [ ]:
df.to_csv('updated_train.csv', index=False)
df = pd.read_csv('updated_train.csv')

## **Exercise 4: Outlier Detection and Handling**

**Goal:** Detect and cap or transform outliers in columns like Fare and Age.  

1. Visualize distributions using boxplots or histograms to identify potential outliers.  
1. Use IQR or Z-score methods to detect them.

**Handle outliers with:**  
+ Quantile capping (e.g. 0.98)
+ Log transformation
+ Row removal

**Compare the dataset before and after treatment.**  

**Note:** Small differences between 0.98 and 0.99 quantiles are normal when extreme values are rare or far apart. Use df.quantile() to explore and choose thresholds empirically, backed by visualization.

In [ ]:
import matplotlib.pyplot as plt
df['Fare'].hist(bins=50)
plt.title('Fare Histrogram')
plt.show()
df.boxplot(column=['Fare'])
plt.title('Fare Boxplot')
plt.show()

**Outlier Detection: IQR Method**

In [ ]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [ ]:
outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
num_outliers= len(outliers)
print('Number of outliers within column "Fare":', num_outliers)

**Outlier Handling: Quantile Capping (Winsorization)**  
Upper: 0.98  
Lower: 0.02

In [ ]:
initial_fare = df['Fare'].describe()
print('Fare BEFORE capping:', initial_fare)

In [ ]:
fare_lower = df['Fare'].quantile(0.02)
fare_upper = df['Fare'].quantile(0.98)
df['Fare'] = df['Fare'].clip(fare_lower, fare_upper)
print('Fare AFTER capping')
print(df['Fare'].describe())

**Outlier Handling: Log transformation**

In [ ]:
import numpy as np
df['Fare_log'] = np.log(df['Fare'])

**Outlier Handling: Row removal**  
This was not used due to the size of the dataset (being already small) and the risk of losing essential information and thus reducing already limited data

## **Exercise 5: Data Normalization**

**Goal:** Scale numerical features to prepare for modeling.

Use StandardScaler (mean = 0, std = 1) for normally distributed features.  
Use MinMaxScaler (range [0, 1]) for features that are skewed or bounded.  

**Important:** Perform this step after outlier treatment to avoid distortion caused by extreme values.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
fare_scaler = MinMaxScaler()
df['Fare'] = fare_scaler.fit_transform(df[['Fare']])
df[['Fare']].head()

## **Exercise 6: Feature Encoding**

**Goal:** Finalize categorical variable encoding.  

Identify remaining categorical columns (e.g. Sex, Embarked, Title).  

**Apply:**  
1. One-Hot Encoding for nominal variables.  
1. Label Encoding if any ordinal variables remain.  

Merge encoded columns back into the main dataset.

**Reminder:** Encoding comes after handling missing values and outliers, but before scaling (if applicable).

In [ ]:
df.head()

**Categorical columns:**
1. Pclass
1. Sex
1. Embarked
1. Title

Column: Pclass  
Encoding: Ordinal

Column: Sex  
Encoding: One-Hot  

Column: Embarked  
Encoding: One-Hot  

Column: Title  
Encoding: Ordinal

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

**Encoding relevant columns**

In [ ]:
ordinal_cols = ['Pclass', 'Title']
one_hot_cols = ['Sex', 'Embarked']
column_transform = ColumnTransformer(
    transformers=[
        ("oe", OrdinalEncoder(), ordinal_cols),
        ("ohe", OneHotEncoder(), one_hot_cols)
    ],
    remainder='drop'
)

**Fit + Transform + Merge**

In [ ]:
encoded = column_transform.fit_transform(df)
encoded_cols = column_transform.get_feature_names_out()
df_encoded = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)
df_final = pd.concat([df, df_encoded], axis=1)
df_final.to_csv("processed_dataset.csv", index=False)
df_final.head()

## **Exercise 7: Data Transformation for Age Feature**

**Goal:** Create and encode age groups.  

+ Use pd.cut() to create bins for life stages (e.g. child, teen, adult, senior).
+ Apply one-hot encoding using pd.get_dummies().

**Example:** You might define bins like [0, 12, 18, 60, 100] and label them accordingly.

In [ ]:
df = pd.read_csv('train.csv')
df.head()

In [ ]:
df['Age'].isna().any()

**Since empty values for 'Age' was detected earlier:**

In [ ]:
df = df.dropna(subset=['Age'])
len(df['Age'])

In [ ]:
df['LifeStage'] = pd.cut(
    df['Age'],
    bins=[0, 12, 18, 60, 120],
    labels=['Child', 'Teen', 'Adult', 'Senior']
)
df.to_csv('age_feature.csv', index=False)
df = pd.read_csv('age_feature.csv')
df.head()

In [ ]:
df_encoded = pd.get_dummies(df, columns=['LifeStage'])
df_encoded.head()

In [ ]:
df.to_csv('age_feature.csv', index=False)